<a href="https://colab.research.google.com/github/mansib-collab/Copilot_Projects/blob/main/Hands_on_with_Tensor_Flow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# __Hands-on with TensorFlow: Part A__

Let's see how TensorFlow works and build a deep neural network model using the MNIST dataset.

## Steps to be followed:
1. Import TensorFlow
2. Load the MNIST dataset
3. Create the model
4. Get predictions from the model
5. Apply softmax activation to the predictions
6. Define the loss function
7. Compile the model
8. Train the model
9. Evaluate the model
10. Create a probability model

## Step 1: Import TensorFlow

- Import TensorFlow and check its version.


In [2]:
import os
# Suppress TensorFlow info and warning messages for cleaner output
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf
# Display the currently installed version of TensorFlow
print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


## Step 2: Load the MNIST dataset

- Load the MNIST dataset and normalize the input data by dividing the train and test sets by 255. The pixel values of the images are divided by 255.0 to normalize them to the range [0, 1]


In [3]:
# Load the built-in MNIST dataset (handwritten digits)
mnist = tf.keras.datasets.mnist

# Split data into training and testing sets
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize pixel values from [0, 255] to [0, 1] to improve model training convergence
x_train, x_test = x_train / 255.0, x_test / 255.0

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [3]:
# x_train

## Step 3: Create the model

- Create a Sequential model with flatten, dense, and dropout layers.


In [4]:
# Define the model structure as a stack of layers
model = tf.keras.models.Sequential([

  # Flatten the 28x28 images into a 1D vector of 784 pixels
  tf.keras.layers.Flatten(input_shape=(28, 28)),

  # Hidden dense layer with 128 neurons and ReLU activation
  tf.keras.layers.Dense(128, activation='relu'),

  # Randomly drop 20% of neurons during training to prevent overfitting
  tf.keras.layers.Dropout(0.2),

  # Final output layer with 10 neurons (one for each digit 0-9)
  tf.keras.layers.Dense(10)
])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


## Step 4: Get predictions from the model

- Get the predictions from the model using the train data for one column, as the optimizer is not yet applied.


In [6]:
# Feed one sample into the untrained model to see initial 'logits' (raw outputs)
predictions = model(x_train[:1]).numpy()
predictions

array([[-0.591321  , -0.5632659 ,  1.1461985 , -0.06361976, -0.1537269 ,
        -0.06594561,  0.57647246,  0.28706315,  0.5878287 , -0.17378634]],
      dtype=float32)

## Step 5: Apply softmax activation to the predictions

- Apply softmax activation to the predictions and print the output in terms of probabilities.


In [8]:
# Softmax converts raw logits into probabilities that sum up to 1.0
tf.nn.softmax(predictions).numpy()

array([[0.0434056 , 0.04464059, 0.24668394, 0.07357385, 0.06723423,
        0.07340292, 0.13954428, 0.10447758, 0.14113799, 0.06589898]],
      dtype=float32)

## Step 6: Define the loss function

- Create a loss function for the model.
- Define the SparseCategoricalCrossentropy loss function.


In [9]:
# Loss function measures how far the model's output is from the actual target
# 'from_logits=True' indicates the model output is raw values, not probabilities
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

# Calculate initial loss for one sample
loss_fn(y_train[:1], predictions).numpy()

np.float32(2.6117916)

## Step 7: Compile the model

- Compile the model with the Adam optimizer, the loss function, and the accuracy metric.

In [10]:
# Configure the training process
model.compile(optimizer='adam', # Optimizer that adjusts weights based on loss
              loss=loss_fn,     # The loss function defined above
              metrics=['accuracy']) # Metric to track performance

## Step 8: Train the model

- Fit the model using the training data for 5 epochs.


In [12]:
# Start training: perform 5 passes (epochs) over the entire dataset
model.fit(x_train, y_train, epochs=5)

Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9699 - loss: 0.0987
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9747 - loss: 0.0825
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9778 - loss: 0.0703
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9799 - loss: 0.0637
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9819 - loss: 0.0566


## Step 9: Evaluate the model

- Evaluate the model using the testing data and print the results.


In [13]:
# Check model performance on unseen test data
model.evaluate(x_test,  y_test, verbose=2)

313/313 - 1s - 2ms/step - accuracy: 0.9794 - loss: 0.0711


[0.07108558714389801, 0.9793999791145325]

## Step 10: Create a probability model

- Create a probability model by adding the softmax layer to the existing model.


In [14]:
# Create a new model that includes the softmax layer to output actual probabilities
probability_model = tf.keras.Sequential([
  model,
  tf.keras.layers.Softmax()
])

# Perform predictions on the first 3 test images
probability_model(x_test[:3])

<tf.Tensor: shape=(3, 10), dtype=float32, numpy=
array([[3.66112007e-09, 1.78330670e-10, 8.06421880e-08, 1.13674705e-05,
        3.58541807e-11, 3.61403387e-08, 6.03987737e-12, 9.99987006e-01,
        4.97595194e-11, 1.54097927e-06],
       [2.79464121e-15, 1.91431718e-06, 9.99997616e-01, 5.25741370e-07,
        1.85877938e-23, 1.21483795e-10, 1.80603118e-13, 7.42818079e-15,
        2.73519905e-11, 5.71172777e-21],
       [1.38064593e-07, 9.99511361e-01, 7.94278167e-05, 7.64960441e-06,
        1.19019242e-05, 4.02867954e-06, 6.89345097e-06, 3.37375910e-04,
        4.09690765e-05, 9.60991784e-08]], dtype=float32)>